**Role in the paper.** Fig. 2: annual cycle of simulated and observed UHIN and the UHIN bias in Rennes for the seven methods. Reads `results/data/Rennes/UHI_Rennes_annualcycle_method_comparison_tasmin.csv` (written by `Eurocities_urclimmask_Rennes_.ipynb`) and saves `results/figures/Rennes/bias_spread_rennes.png`.

# Fig. 2 readability variants (Steven #42, #43)

**D** (recommended): original layout; every simulated line shares one style (thick short dashes, no markers), the three identical a2a curves cycle their three colors within that style; observed lines are thin long dashes with markers, interleaved in two colors where a2a and p2p share the curve. C = previous step, 0 = current, A/B = earlier, more reductive options.

In [ ]:
# Fig. 2 (Rennes annual UHIN cycle) - readability variants for Steven's comments #42/#43.
# Reads the method-comparison CSV written by Eurocities_urclimmask_Rennes_.ipynb (cell 49).

import sys, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from pathlib import Path
ROOT = Path.cwd().parents[1]  # run from scripts/plotting
csv = ROOT / 'results/data/Rennes/UHI_Rennes_annualcycle_method_comparison_tasmin.csv'
outdir = ROOT / 'results/figures/Rennes'


In [ ]:
df = pd.read_csv(csv, index_col=0)
S = "UHIN"
MONTHS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
x = np.arange(12)
# paper name -> csv key
M = {'p2p_UCN_uc':'p2p_un_ucdb','p2p_UCN_lcz':'p2p_un_lcz','p2p_MS_uc':'p2p_ms_ucdb',
     'a2a_UCN_uc':'urclim_un_ucdb','a2a_UCN_lcz':'urclim_un_lcz','a2a_MS_uc':'urclim_ms_ucdb',
     'p2a_MS_dist':'blend_ms_ucdb'}
g = lambda m, k: df[f"{S}_{M[m]}_{k}"].values

def layout(title):
    fig = plt.figure(figsize=(18, 12))
    gs = fig.add_gridspec(2, 2)
    a1, a2, a3 = fig.add_subplot(gs[0,0]), fig.add_subplot(gs[0,1]), fig.add_subplot(gs[1,:])
    p = a3.get_position(); a3.set_position([p.x0+0.05, p.y0, p.width-0.20, p.height])
    for a, t in [(a1,'simulated UHIN'),(a2,'observed UHIN'),(a3,'UHIN bias')]:
        a.set_title(t, fontsize=16); a.set_xticks(x); a.set_xticklabels(MONTHS, fontsize=12)
        a.grid(color='0.88', lw=0.8); a.set_axisbelow(True)
    a1.sharey(a2)
    a1.set_ylabel('UHIN [°C]', fontsize=14); a3.set_ylabel('UHIN bias [°C]', fontsize=14); a3.set_xlabel('Month', fontsize=14)
    fig.suptitle(title, fontsize=13, x=0.02, ha='left', color='0.35')
    return fig, a1, a2, a3

def sep_line(a3, col=None):
    col = col or COL0
    a3.axvline(8, color='black', lw=1.2)
    # September bias values next to the vertical line (as in the original cell 55)
    for m,xo,yo in (('p2p_MS_uc',8.2,-0.8),('a2a_UCN_uc',8.1,-0.225),('p2a_MS_dist',8.1,1.86)):
        a3.text(xo, yo, f"{g(m,'bias')[8]:.2f}", color=col[m], fontsize=12, va='bottom')
    a3.axhline(0, color='green', lw=3, marker='o')

# ---------------- current scheme (approximate copy of cell 55) ----------------
COL0 = {'p2a_MS_dist':'darkred','a2a_UCN_lcz':'firebrick','a2a_MS_uc':'crimson','a2a_UCN_uc':'red',
        'p2p_UCN_uc':'orangered','p2p_UCN_lcz':'darkorange','p2p_MS_uc':'salmon'}
def current():
    fig,a1,a2,a3 = layout('0 - current scheme (7 reds/oranges, faded counterpart lines)')
    for m,c in COL0.items():
        a1.plot(x, g(m,'obs'), color=c, ls=(0,(5,7)), marker='o', lw=4, alpha=0.2)
        a1.plot(x, g(m,'sim'), color=c, lw=4)
        a2.plot(x, g(m,'sim'), color=c, lw=4, alpha=0.2)
        a2.plot(x, g(m,'obs'), color=c, ls=(0,(5,7)), marker='o', lw=4)
        a3.plot(x, g(m,'bias'), color=c, lw=4, ls=(0,(3,1,1,1)))
    for a in (a1,a2): a.plot(x, np.zeros(12), color='green', lw=3, marker='o')
    sep_line(a3)
    h=[Line2D([0],[0],color=c,lw=4,label=m) for m,c in COL0.items()]+[Line2D([0],[0],color='green',lw=4,label='Rural ref')]
    a3.legend(handles=h, title='Evaluation method', loc='center left', bbox_to_anchor=(1.02,0.5), fontsize=13)
    return fig

# ---------------- variant A: keep colors, remove the overlaps ----------------
def variant_a():
    fig,a1,a2,a3 = layout('A - same colors; identical curves drawn once, no faded counterparts')
    for m in ['p2p_UCN_uc','p2p_UCN_lcz','p2p_MS_uc','p2a_MS_dist']:
        a1.plot(x, g(m,'sim'), color=COL0[m], lw=3, marker='o', ms=6)
    a1.plot(x, g('a2a_UCN_uc','sim'), color='0.15', lw=3, ls='--', marker='s', ms=6)
    # observed UHIN only depends on network + classification: p2p_X == a2a_X
    obs = {'UCN_uc':('p2p_UCN_uc','red'),'UCN_lcz':('p2p_UCN_lcz','darkorange'),'MS_uc':('p2p_MS_uc','salmon'),'MS_dist':('p2a_MS_dist','darkred')}
    for lab,(m,c) in obs.items():
        a2.plot(x, g(m,'obs'), color=c, lw=3, ls='--', marker='o', ms=6)
    for a in (a1,a2): a.plot(x, np.zeros(12), color='green', lw=2.5)
    for m,c in COL0.items(): a3.plot(x, g(m,'bias'), color=c, lw=3, ls=(0,(3,1,1,1)))
    sep_line(a3)
    h=[Line2D([0],[0],color=c,lw=4,label=m) for m,c in COL0.items()]+[Line2D([0],[0],color='0.15',lw=3,ls='--',marker='s',label='a2a simulated (identical)'),Line2D([0],[0],color='green',lw=3,label='Rural ref')]
    a3.legend(handles=h, title='Evaluation method', loc='center left', bbox_to_anchor=(1.02,0.5), fontsize=13)
    return fig

# ---------------- variant B: composite encoding ----------------
# color = observational set (network + classification), validated warm ramp (CVD-checked, all pairs)
# marker / line style = simulation aggregation
OBSC = {'UCN_uc':'#c8102e','UCN_lcz':'#f26a1b','MS_uc':'#e9b949','MS_dist':'#5c0a12'}
AGG = {'p2p':dict(ls='-',marker='o'),'a2a':dict(ls='--',marker='s'),'p2a':dict(ls=':',marker='D')}
def split(m):
    agg, net, cls = m.split('_'); return agg, f'{net}_{cls}'
def variant_b():
    fig,a1,a2,a3 = layout('B - color = observation set (network+classification), line/marker = aggregation')
    kw = dict(lw=2.5, ms=7, mec='white', mew=1)
    for m in ['p2p_UCN_uc','p2p_UCN_lcz','p2p_MS_uc','p2a_MS_dist']:
        agg,o = split(m); a1.plot(x, g(m,'sim'), color=OBSC[o], **AGG[agg], **kw)
    a1.plot(x, g('a2a_UCN_uc','sim'), color='0.2', **AGG['a2a'], **kw)
    for o,c in OBSC.items():
        m = 'p2a_MS_dist' if o=='MS_dist' else f'p2p_{o}'
        a2.plot(x, g(m,'obs'), color=c, ls='-', marker='o', **{k:v for k,v in kw.items()})
    for a in (a1,a2): a.plot(x, np.zeros(12), color='green', lw=2.5)
    for m in M:
        agg,o = split(m); a3.plot(x, g(m,'bias'), color=OBSC[o], **AGG[agg], **kw)
    sep_line(a3)
    h1=[Line2D([0],[0],color=c,lw=3,label=o) for o,c in OBSC.items()]+[Line2D([0],[0],color='0.2',lw=3,label='a2a simulated'),Line2D([0],[0],color='green',lw=3,label='Rural ref')]
    h2=[Line2D([0],[0],color='0.3',lw=2.5,ms=7,label=k,**v) for k,v in AGG.items()]
    l1=a3.legend(handles=h1, title='Network + classification', loc='upper left', bbox_to_anchor=(1.02,1.0), fontsize=13)
    a3.add_artist(l1)
    a3.legend(handles=h2, title='Aggregation', loc='lower left', bbox_to_anchor=(1.02,0.0), fontsize=13)
    return fig


# ---------------- variant C: original figure, distinct a2a colors ----------------
# Same layout and line styles as cell 55; the three a2a colors are now clearly different
# (dark red, red, magenta), so the interleaved dashes of the identical a2a curves show all three.
COLC = {'p2a_MS_dist':'#4a0010','a2a_UCN_lcz':'#a50f15','a2a_MS_uc':'#ef3b2c','a2a_UCN_uc':'#c51b7d',
        'p2p_UCN_uc':'#ff7f00','p2p_UCN_lcz':'#ffd23f','p2p_MS_uc':'#fcae91'}
def variant_c(col=COLC, title='C - original layout, interleaved dashes, distinct a2a colors'):
    fig,a1,a2,a3 = layout(title)
    a2a = ['a2a_UCN_uc','a2a_UCN_lcz','a2a_MS_uc']; p2p = ['p2p_UCN_uc','p2p_UCN_lcz','p2p_MS_uc']
    for ax, main in ((a1,'sim'),(a2,'obs')):
        fade = 'obs' if main=='sim' else 'sim'
        al = lambda k: 1.0 if k==main else 0.2
        # observed: a2a and p2p share the curve -> two colors interleaved
        for i,(ma,mp) in enumerate(zip(a2a,p2p)):
            ax.plot(x, g(ma,'obs'), color=col[ma], ls=(0,(5,7)), marker='o', lw=4, alpha=al('obs'))
            ax.plot(x, g(mp,'obs'), color=col[mp], ls=(6,(5,7)), marker='o', lw=4, alpha=al('obs'))
        ax.plot(x, g('p2a_MS_dist','obs'), color=col['p2a_MS_dist'], ls='--', marker='o', lw=4, alpha=al('obs'))
        # simulated: three identical a2a curves -> three colors interleaved, on top
        for i,ma in enumerate(a2a):
            ax.plot(x, g(ma,'sim'), color=col[ma], ls=(4.5*i,(3.2,10.3)), lw=4.5, alpha=al('sim'), zorder=10, solid_capstyle='butt', dash_capstyle='butt')
        for mp in p2p: ax.plot(x, g(mp,'sim'), color=col[mp], lw=4, alpha=al('sim'))
        ax.plot(x, g('p2a_MS_dist','sim'), color=col['p2a_MS_dist'], lw=4, alpha=al('sim'))
        ax.plot(x, np.zeros(12), color='green', lw=3, marker='o')
    for m,c in col.items(): a3.plot(x, g(m,'bias'), color=c, lw=4, ls=(0,(3,1,1,1)))
    sep_line(a3, col)
    h=[Line2D([0],[0],color=c,lw=4,label=m) for m,c in col.items()]+[Line2D([0],[0],color='green',lw=4,label='Rural ref')]
    t=[Line2D([0],[0],color='black',lw=4,ls='-',label='Simulation'),Line2D([0],[0],color='black',lw=4,ls=(0,(3.2,1.3)),label='Simulation a2a (3 identical)'),Line2D([0],[0],color='black',lw=4,ls='--',marker='o',label='Observation'),Line2D([0],[0],color='black',lw=4,ls=(0,(3,1,1,1)),label='Bias')]
    l1=a3.legend(handles=h, title='Evaluation method', loc='center left', bbox_to_anchor=(1.02,0.6), fontsize=14)
    a3.add_artist(l1); a3.legend(handles=t, title='Type', loc='upper left', bbox_to_anchor=(1.02,.25), fontsize=14)
    return fig

def variant_d(col=COLC, title='D - one shared simulated style (short dashes, thick), observed thin long dashes with markers'):
    fig,a1,a2,a3 = layout(title)
    a2a = ['a2a_UCN_uc','a2a_UCN_lcz','a2a_MS_uc']; p2p = ['p2p_UCN_uc','p2p_UCN_lcz','p2p_MS_uc']
    SIM = dict(lw=5, solid_capstyle='butt', dash_capstyle='butt')
    OBS = dict(lw=2.2, marker='o', ms=7, mec='white', mew=0.8, dash_capstyle='butt')
    for ax, main in ((a1,'sim'),(a2,'obs')):
        al = lambda k: 1.0 if k==main else 0.2
        # observed: thin long dashes with markers; a2a and p2p share the curve -> two colors interleaved
        for ma,mp in zip(a2a,p2p):
            ax.plot(x, g(ma,'obs'), color=col[ma], ls=(0,(8,12)), alpha=al('obs'), **OBS)
            ax.plot(x, g(mp,'obs'), color=col[mp], ls=(10,(8,12)), alpha=al('obs'), **OBS)
        ax.plot(x, g('p2a_MS_dist','obs'), color=col['p2a_MS_dist'], ls=(0,(8,2)), alpha=al('obs'), **OBS)
        # simulated: thick short dashes (dash 3, gap 1) for every method; the three identical a2a curves cycle colors
        for mp in p2p: ax.plot(x, g(mp,'sim'), color=col[mp], ls=(0,(3,1)), alpha=al('sim'), **SIM)
        ax.plot(x, g('p2a_MS_dist','sim'), color=col['p2a_MS_dist'], ls=(0,(3,1)), alpha=al('sim'), **SIM)
        for i,ma in enumerate(a2a):
            ax.plot(x, g(ma,'sim'), color=col[ma], ls=(4*i,(3,9)), alpha=al('sim'), zorder=10, **SIM)
        ax.plot(x, np.zeros(12), color='green', lw=3, marker='o')
    for m,c in col.items(): a3.plot(x, g(m,'bias'), color=c, lw=4, ls=(0,(3,1,1,1)))
    sep_line(a3, col)
    h=[Line2D([0],[0],color=c,lw=4,label=m) for m,c in col.items()]+[Line2D([0],[0],color='green',lw=4,label='Rural ref')]
    t=[Line2D([0],[0],color='black',ls=(0,(3,1)),label='Simulation',**SIM),Line2D([0],[0],color='black',ls=(0,(8,2)),label='Observation',**OBS),Line2D([0],[0],color='black',lw=4,ls=(0,(3,1,1,1)),label='Bias')]
    l1=a3.legend(handles=h, title='Evaluation method', loc='center left', bbox_to_anchor=(1.02,0.6), fontsize=14)
    a3.add_artist(l1); a3.legend(handles=t, title='Type', loc='upper left', bbox_to_anchor=(1.02,.25), fontsize=14, handlelength=4)
    return fig



In [ ]:
variant_d();

In [ ]:
variant_c();

In [ ]:
current();

In [ ]:
variant_a();

In [ ]:
variant_b();

In [ ]:
# save the chosen variant for the paper
# fig = variant_d(title=''); fig.savefig(outdir / 'bias_spread_rennes.png', dpi=300, bbox_inches='tight')